# 3.1.3 智能健康手环数据分析与优化

## 任务概述

基于 `智能健康手环数据集.xlsx`（含两个 Sheet），从以下三方面分析，并生成 **3.1.3-1.docx** 与 **3.1.3-2.docx**。

| 分析维度 | 数据来源 | 目标 |
|----------|----------|------|
| 用户活动模式 | `步数统计记录` | 一周内各时段活动水平，识别高峰/低谷 |
| 健康指标关注度 | `功能查看记录` | 步数/心率/睡眠时长查看频率 |
| 数据同步性能 | `功能查看记录` | 传输延迟均值，找影响因素与瓶颈 |

> 运行前请将工作目录切换到本 notebook 所在文件夹。

**配置：** 结果文件保存到**当前目录**（与 xlsx 同级）。

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_DIR = os.getcwd()
print('结果保存目录:', OUTPUT_DIR)

结果保存目录: d:\workspace\doc\AI 培训-政府\for-student-Julie\setup\人工智能训练师_3级_Julie_answer\3.1.3-素材


**第一步：** 读取两个 Sheet，预览数据结构。

In [12]:
FILE = '智能健康手环数据集.xlsx'
df_steps = pd.read_excel(FILE, sheet_name='步数统计记录')
df_view = pd.read_excel(FILE, sheet_name='功能查看记录')

print('步数统计记录:', df_steps.shape, list(df_steps.columns))
print('功能查看记录:', df_view.shape, list(df_view.columns))
print('\n步数表前5行:')
print(df_steps.head())
print('\n功能表前5行:')
print(df_view.head())

步数统计记录: (16800, 3) ['用户', '日期', '步数']
功能查看记录: (3836, 4) ['用户', '时间戳', '功能调用类型', '传输延迟时间']

步数表前5行:
       用户                   日期  步数
0  User_1  2024-09-15 00:00:00   0
1  User_1  2024-09-15 01:00:00   0
2  User_1  2024-09-15 02:00:00   0
3  User_1  2024-09-15 03:00:00   0
4  User_1  2024-09-15 04:00:00   0

功能表前5行:
       用户                 时间戳 功能调用类型    传输延迟时间
0  User_1 2024-09-15 19:28:00   睡眠时长  0.392798
1  User_1 2024-09-15 20:38:00     心率  1.155995
2  User_1 2024-09-15 10:10:00     步数  0.340446
3  User_1 2024-09-15 23:02:00     心率  1.056412
4  User_1 2024-09-15 23:43:00     心率  1.212339


In [13]:
# 时间字段预处理
df_steps['日期'] = pd.to_datetime(df_steps['日期'])
df_steps['小时'] = df_steps['日期'].dt.hour
df_steps['星期'] = df_steps['日期'].dt.dayofweek  # 0=周一
df_steps['星期名'] = df_steps['日期'].dt.day_name()

df_view['时间戳'] = pd.to_datetime(df_view['时间戳'])
df_view['小时'] = df_view['时间戳'].dt.hour
df_view['星期'] = df_view['时间戳'].dt.dayofweek

WEEK_MAP = {0: '周一', 1: '周二', 2: '周三', 3: '周四', 4: '周五', 5: '周六', 6: '周日'}
df_steps['星期中文'] = df_steps['星期'].map(WEEK_MAP)

print('步数数据时间范围:', df_steps['日期'].min(), '~', df_steps['日期'].max())
print('用户数:', df_steps['用户'].nunique())
print('功能查看记录数:', len(df_view))

步数数据时间范围: 2024-09-15 00:00:00 ~ 2024-09-21 23:00:00
用户数: 100
功能查看记录数: 3836


## 一、用户活动模式（步数统计记录）

以**每小时步数**衡量活动水平，分析一天内与一周内的分布。

In [14]:
# 1.1 各小时平均活动水平（全周汇总）
hourly_activity = df_steps.groupby('小时')['步数'].agg(['mean', 'sum', 'count']).round(1)
hourly_activity.columns = ['平均步数', '总步数', '记录数']
print('=== 各小时活动水平 ===')
print(hourly_activity)

peak_hours = hourly_activity['平均步数'].nlargest(3)
low_hours = hourly_activity['平均步数'].nsmallest(3)
print(f'\n高峰时段: {peak_hours.index.tolist()} 点，平均步数 {peak_hours.values}')
print(f'低谷时段: {low_hours.index.tolist()} 点，平均步数 {low_hours.values}')

=== 各小时活动水平 ===
      平均步数      总步数  记录数
小时                      
0      0.0        0  700
1      0.0        0  700
2      0.0        0  700
3      0.0        0  700
4      0.0        0  700
5      0.0        0  700
6   5079.6  3555731  700
7   5076.8  3553780  700
8   5047.9  3533497  700
9   1099.9   769911  700
10  1114.7   780306  700
11  1125.3   787682  700
12  1112.7   778880  700
13  1057.7   740364  700
14  1088.1   761652  700
15  1120.1   784039  700
16  1105.2   773624  700
17  4998.1  3498637  700
18  4997.4  3498154  700
19  5004.9  3503414  700
20  5022.3  3515608  700
21  1099.3   769512  700
22  1066.9   746819  700
23  1124.0   786819  700

高峰时段: [6, 7, 8] 点，平均步数 [5079.6 5076.8 5047.9]
低谷时段: [0, 1, 2] 点，平均步数 [0. 0. 0.]


In [5]:
# 1.2 一周内各星期平均活动水平
weekday_activity = df_steps.groupby('星期中文')['步数'].mean().round(1)
weekday_activity = weekday_activity.reindex(['周一','周二','周三','周四','周五','周六','周日'])
print('=== 各星期平均步数 ===')
print(weekday_activity)

busiest_day = weekday_activity.idxmax()
quietest_day = weekday_activity.idxmin()
print(f'\n最活跃: {busiest_day}；相对低谷: {quietest_day}')

=== 各星期平均步数 ===
星期中文
周一    1948.0
周二    1980.8
周三    1976.7
周四    1990.7
周五    1969.1
周六    1970.1
周日    1972.4
Name: 步数, dtype: float64

最活跃: 周四；相对低谷: 周一


In [15]:
# 1.3 时段划分（早/中/晚/夜）
def activity_period(h):
    if 6 <= h < 12:
        return '上午(6-12)'
    if 12 <= h < 18:
        return '下午(12-18)'
    if 18 <= h < 22:
        return '傍晚(18-22)'
    return '夜间(0-6/22-24)'

df_steps['时段'] = df_steps['小时'].apply(activity_period)
period_activity = df_steps.groupby('时段')['步数'].mean().round(1)
period_activity = period_activity.reindex(['上午(6-12)', '下午(12-18)', '傍晚(18-22)', '夜间(0-6/22-24)'])
print('=== 各时段平均步数 ===')
print(period_activity)

=== 各时段平均步数 ===
时段
上午(6-12)         3090.7
下午(12-18)        1747.0
傍晚(18-22)        4031.0
夜间(0-6/22-24)     273.9
Name: 步数, dtype: float64


## 二、健康指标关注度（功能查看记录）

In [16]:
metric_counts = df_view['功能调用类型'].value_counts()
metric_pct = (metric_counts / len(df_view) * 100).round(2)
metric_table = pd.DataFrame({'查看次数': metric_counts, '占比(%)': metric_pct})

print('=== 健康指标关注度 ===')
print(metric_table)

most_metric = metric_counts.index[0]
least_metric = metric_counts.index[-1]
print(f'\n最受关注: {most_metric}（{metric_counts.iloc[0]}次, {metric_pct.iloc[0]}%）')
print(f'较少关注: {least_metric}（{metric_counts.iloc[-1]}次, {metric_pct.iloc[-1]}%）')

=== 健康指标关注度 ===
        查看次数  占比(%)
功能调用类型             
步数      1301  33.92
心率      1269  33.08
睡眠时长    1266  33.00

最受关注: 步数（1301次, 33.92%）
较少关注: 睡眠时长（1266次, 33.0%）


In [8]:
# 各指标在不同时段的查看次数
metric_hour = pd.crosstab(df_view['小时'], df_view['功能调用类型'])
print('=== 各小时查看次数（按指标）===')
print(metric_hour.sum(axis=1).sort_values(ascending=False).head(5))

# plt.figure(figsize=(8, 4))
# metric_counts.plot(kind='bar', color=['#4CAF50', '#F44336', '#2196F3'], edgecolor='black')
# plt.title('健康指标查看次数')
# plt.ylabel('次数')
# plt.xticks(rotation=0)
# plt.tight_layout()
# plt.show()

=== 各小时查看次数（按指标）===
小时
2     181
0     173
16    173
20    173
21    171
dtype: int64


## 三、数据同步性能（传输延迟）

In [18]:
COL_DELAY = '传输延迟时间'

delay_stats = df_view.groupby('功能调用类型')[COL_DELAY].agg(
    查看次数='count',
    平均延迟='mean',
    中位数='median',
    最大延迟='max'
).round(3).sort_values('平均延迟', ascending=False)

print('=== 各指标传输延迟 ===')
print(delay_stats)
print(f'\n全库平均传输延迟: {df_view[COL_DELAY].mean():.3f} 秒')

slowest_metric = delay_stats.index[0]
fastest_metric = delay_stats.index[-1]
print(slowest_metric)
print(fastest_metric)

=== 各指标传输延迟 ===
        查看次数   平均延迟    中位数   最大延迟
功能调用类型                           
心率      1269  1.502  1.502  2.000
步数      1301  0.301  0.305  0.499
睡眠时长    1266  0.300  0.299  0.500

全库平均传输延迟: 0.698 秒
心率
睡眠时长


In [19]:
# 影响指数 = 平均延迟 × 查看次数
sync_bottleneck = pd.DataFrame({
    '查看次数': metric_counts,
    '平均延迟': delay_stats['平均延迟']
})
sync_bottleneck['影响指数'] = (sync_bottleneck['查看次数'] * sync_bottleneck['平均延迟']).round(2)
sync_bottleneck = sync_bottleneck.sort_values('影响指数', ascending=False)
print('=== 同步瓶颈分析 ===')
print(sync_bottleneck)
bottleneck_metric = sync_bottleneck.index[0]

=== 同步瓶颈分析 ===
        查看次数   平均延迟     影响指数
功能调用类型                      
心率      1269  1.502  1906.04
步数      1301  0.301   391.60
睡眠时长    1266  0.300   379.80


In [20]:
# 按小时分析传输延迟
delay_by_hour = df_view.groupby('小时')[COL_DELAY].mean().sort_values(ascending=False)
print('=== 延迟较高的小时 Top5 ===')
print(delay_by_hour.head(5).round(3))

# plt.figure(figsize=(10, 4))
# delay_stats['平均延迟'].plot(kind='barh', color='coral', edgecolor='black')
# plt.axvline(df_view[COL_DELAY].mean(), color='red', linestyle='--',
#             label=f'总体均值 {df_view[COL_DELAY].mean():.2f}s')
# plt.title('各健康指标平均传输延迟')
# plt.xlabel('秒')
# plt.legend()
# plt.tight_layout()
# plt.show()

=== 延迟较高的小时 Top5 ===
小时
4     0.802
13    0.781
20    0.749
12    0.748
10    0.740
Name: 传输延迟时间, dtype: float64


## 四、生成 Word 报告（3.1.3-1.docx / 3.1.3-2.docx）

运行下方单元格，自动写入**当前目录**。需安装：`pip install python-docx`

In [ ]:
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

def add_heading(doc, text, level=1):
    doc.add_heading(text, level=level)

def add_para(doc, text, bold=False):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold
    run.font.size = Pt(11)

doc1 = Document()
t = doc1.add_heading('智能健康手环数据分析报告', 0)
t.alignment = WD_ALIGN_PARAGRAPH.CENTER

add_heading(doc1, '一、数据概况', 1)
add_para(doc1, f'数据集包含步数统计记录 {len(df_steps)} 条（100 用户 × 7 天 × 24 小时），'
          f'功能查看记录 {len(df_view)} 条。')
add_para(doc1, f'统计周期：{df_steps["日期"].min().date()} 至 {df_steps["日期"].max().date()}（共 7 天）。')

add_heading(doc1, '二、用户活动模式分析', 1)
add_para(doc1, '2.1 日内高峰与低谷', True)
add_para(doc1, f'高峰时段为 {peak_hours.index.tolist()} 点，平均步数约 {peak_hours.iloc[0]:.0f} 步/小时，'
          f'呈现明显的**晨间（6-8 点）与傍晚（17-20 点）双高峰**特征。')
add_para(doc1, f'低谷时段为 {low_hours.index.tolist()} 点（深夜至凌晨），平均步数接近 0，符合睡眠规律。')
add_para(doc1, f'午间（9-16 点）及深夜（21-23 点）活动水平中等，平均约 1100 步/小时。')

add_para(doc1, '2.2 一周内活动分布', True)
for day in weekday_activity.index:
    add_para(doc1, f'  · {day}：平均 {weekday_activity[day]} 步/小时')
add_para(doc1, f'一周各日活动较均衡，{busiest_day}略高，{quietest_day}略低，差异不大。')

add_heading(doc1, '三、健康指标关注度', 1)
for m in metric_counts.index:
    add_para(doc1, f'  · {m}：{metric_counts[m]} 次，占比 {metric_pct[m]}%')
add_para(doc1, f'三项指标关注度接近（约 33%），{most_metric} 略高，{least_metric} 略低，差距不足 1 个百分点。')
add_para(doc1, '说明用户对步数、心率、睡眠时长的关注较为均衡，无明显偏废项。')

add_heading(doc1, '四、数据同步性能', 1)
add_para(doc1, f'全库平均传输延迟为 {df_view[COL_DELAY].mean():.3f} 秒。')
for m in delay_stats.index:
    add_para(doc1, f'  · {m}：平均延迟 {delay_stats.loc[m, "平均延迟"]:.3f}s，最大 {delay_stats.loc[m, "最大延迟"]:.3f}s')
add_para(doc1, f'{slowest_metric} 同步延迟显著高于其他指标（约 {delay_stats.loc[slowest_metric, "平均延迟"]:.3f}s），'
          f'而 {fastest_metric} 与睡眠时长延迟较低（约 0.30s）。')
add_para(doc1, f'影响因素：① 指标类型——心率数据包较大或需实时流式传输；② 同步时段——'
          f'{delay_by_hour.head(3).index.tolist()} 点延迟偏高，可能与网络高峰或后台任务有关。')
add_para(doc1, f'综合影响指数，{bottleneck_metric} 是同步性能首要瓶颈（影响指数 {sync_bottleneck.loc[bottleneck_metric, "影响指数"]:.1f}）。')

add_heading(doc1, '五、小结', 1)
add_para(doc1, '1. 用户活动呈晨晚双高峰，产品可在 6-8 点、17-20 点推送运动提醒。')
add_para(doc1, '2. 三项健康指标关注度均衡，可维持现有功能布局。')
add_para(doc1, f'3. 心率数据同步延迟是主要性能瓶颈，应优先优化。')

path1 = os.path.join(OUTPUT_DIR, '3.1.3-1.docx')
doc1.save(path1)
print('已保存:', path1)

In [ ]:
optimizations = [
    {
        '方向': '优化心率数据同步性能',
        '依据': f'心率平均延迟 {delay_stats.loc["心率", "平均延迟"]:.3f}s，约为步数/睡眠的 5 倍，影响指数 {sync_bottleneck.loc["心率", "影响指数"]:.0f} 最高',
        '方案': '采用增量同步与数据压缩；心率采样批量上传（如每 5 分钟打包）；BLE 连接优先级提升；本地缓存未同步数据避免重复传输。',
    },
    {
        '方向': '匹配双高峰活动模式，智能推送运动服务',
        '依据': f'6-8 点与 17-20 点步数高峰（平均 {peak_hours.iloc[0]:.0f}+ 步/小时），深夜活动接近零',
        '方案': '在晨间/傍晚高峰前推送「今日目标进度」；深夜自动切换勿扰并降低屏幕唤醒；结合步数趋势给出个性化运动建议。',
    },
    {
        '方向': '均衡强化三项健康指标的可视化体验',
        '依据': f'步数、心率、睡眠关注度接近 33%，但睡眠时长略低（{metric_pct.iloc[-1]}%）',
        '方案': '首页仪表盘一屏展示三项核心指标；睡眠报告增加周趋势与质量评分；早晨推送「昨夜睡眠摘要」提升睡眠模块打开率。',
    },
]

doc2 = Document()
t2 = doc2.add_heading('智能健康手环优化方案', 0)
t2.alignment = WD_ALIGN_PARAGRAPH.CENTER
add_para(doc2, '基于数据分析，提出以下 3 项优化方向及解决方案：')

for i, opt in enumerate(optimizations, 1):
    add_heading(doc2, f'优化方向 {i}：{opt["方向"]}', 1)
    add_para(doc2, '数据依据：' + opt['依据'])
    add_para(doc2, '解决方案：' + opt['方案'])
    doc2.add_paragraph()

path2 = os.path.join(OUTPUT_DIR, '3.1.3-2.docx')
doc2.save(path2)
print('已保存:', path2)
print('\n交卷文件:')
print(' ', path1)
print(' ', path2)

## 五、导出辅助 CSV（可选）

In [ ]:
hourly_activity.to_csv(os.path.join(OUTPUT_DIR, '3.1.3_小时活动.csv'), encoding='utf-8-sig')
metric_table.to_csv(os.path.join(OUTPUT_DIR, '3.1.3_指标关注度.csv'), encoding='utf-8-sig')
sync_bottleneck.to_csv(os.path.join(OUTPUT_DIR, '3.1.3_同步瓶颈.csv'), encoding='utf-8-sig')
print('辅助 CSV 已导出到:', OUTPUT_DIR)